# AMML 2026 — финальное решение


Два Jina v3.5 QLoRA-reranker обучаются на всех размеченных парах. Итоговый прогноз — среднее их рангов внутри запроса.


## 1. Импорты и настройки


In [ ]:
# Final AMML 2026 solution: Jina v3.5 mixed-loss + LambdaRank rank blend.
import gc, os, random
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, BitsAndBytesConfig, get_linear_schedule_with_warmup

SEED, MODEL = 2026, 'jinaai/jina-reranker-v3.5'
MAX_CONTEXT, MAX_QUERY, MAX_DOC, ACCUM, LR = 6144, 128, 384, 4, 5e-5
INSTRUCTION = ('Rank Russian web-search passages by their relevance to the query. '
               'A relevant passage directly answers the query or fulfils its intent.')

def seed_all():
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

def find_data():
    candidates = [Path.cwd(), Path.cwd() / 'amml-ranking-competition-2026']
    if os.environ.get('AMML_DATA_DIR'): candidates.insert(0, Path(os.environ['AMML_DATA_DIR']))
    kaggle = Path('/kaggle/input')
    if kaggle.exists(): candidates += [p.parent for p in kaggle.rglob('train.csv')]
    for p in candidates:
        if (p / 'train.csv').is_file() and (p / 'test.csv').is_file(): return p
    raise FileNotFoundError('Set AMML_DATA_DIR or provide train.csv and test.csv; see README.md.')

def clean(s): return s.fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
def docs(g): return (clean(g.title) + ' [SEP] ' + clean(g.text)).tolist()



## 2. Listwise-вход для Jina и инициализация QLoRA


In [ ]:
def prompt(query, documents, special):
    system = ('<|im_start|>system\nYou are a search relevance expert who ranks passages by relevance to a query. '
              'For questions, prefer passages that answer the question directly. Follow the supplied instruction.\n'
              '<|im_end|>\n<|im_start|>user\n')
    body = f'I will provide {len(documents)} passages. Rank them for the query: {query}\n'
    body += f'<instruct>\n{INSTRUCTION}\n</instruct>\n'
    body += '\n'.join(f'<passage id="{i}">\n{d}{special["doc_embed_token"]}\n</passage>' for i, d in enumerate(documents))
    return system + body + f'\n<query>\n{query}{special["query_embed_token"]}\n</query><|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

def inputs(tok, group, special):
    q = clean(group['query']).iloc[0]
    q = tok.decode(tok(q, truncation=True, max_length=MAX_QUERY, add_special_tokens=False).input_ids, skip_special_tokens=False)
    budget = max(64, (MAX_CONTEXT - MAX_QUERY - 1024 - 32 * len(group)) // len(group))
    ds = []
    for d in docs(group):
        ids = tok(d, truncation=True, max_length=min(MAX_DOC, budget), add_special_tokens=False).input_ids
        ds.append(tok.decode(ids, skip_special_tokens=False))
    return tok(prompt(q, ds, special), truncation=True, max_length=MAX_CONTEXT, return_tensors='pt')

def make_model():
    tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, padding_side='left')
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    model = AutoModel.from_pretrained(MODEL, trust_remote_code=True, quantization_config=quant, torch_dtype=torch.bfloat16, device_map={'': 0})
    model.config.use_cache = False; special = dict(model.special_tokens)
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM', target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
    return tok, get_peft_model(model, lora), special



## 3. Функции потерь и инференс по запросам


In [ ]:
def loss_fn(scores, labels, kind):
    scale = 10.0; bce = torch.nn.functional.binary_cross_entropy_with_logits(scores * scale, labels)
    pos, neg = torch.where(labels > .5)[0], torch.where(labels < .5)[0]
    if len(pos) == 0 or len(neg) == 0: return bce
    pair = -torch.nn.functional.logsigmoid(scale * (scores[pos][:, None] - scores[neg][None, :]))
    if kind == 'mixed': return .5 * bce + .5 * pair.mean()
    with torch.no_grad():
        order = torch.argsort(scores, descending=True); rank = torch.empty_like(order); rank[order] = torch.arange(1, len(scores) + 1, device=scores.device)
        discount = torch.where(rank <= 10, 1 / torch.log2(rank.float() + 1), torch.zeros_like(rank, dtype=torch.float))
        npos = min(10, int(labels.sum().item())); idcg = (1 / torch.log2(torch.arange(2, npos + 2, device=scores.device, dtype=torch.float))).sum()
        delta = (discount[pos][:,None] - discount[neg][None,:]).abs() / idcg.clamp_min(1e-8)
    return .2 * bce + .8 * (pair * delta).sum() / delta.sum().clamp_min(1e-8)

@torch.inference_mode()
def infer(model, tok, frame, special, name):
    out = np.empty(len(frame), dtype=np.float32); model.eval(); device = next(model.parameters()).device
    for _, g in tqdm(frame.groupby('query_id', sort=False), desc=name, unit='query'):
        x = {k:v.to(device) for k,v in inputs(tok, g, special).items()}
        score = model(**x).scores.view(-1).float().cpu().numpy()
        if len(score) != len(g): raise RuntimeError('Unexpected number of Jina scores.')
        out[g.index.to_numpy()] = score
    return out



## 4. Обучение двух финальных моделей


In [ ]:
def fit_predict(kind, steps):
    seed_all(); tok, model, special = make_model(); device = next(model.parameters()).device
    groups = [g for _, g in train.groupby('query_id', sort=False)]
    opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=.01)
    sch = get_linear_schedule_with_warmup(opt, max(1, steps // 20), steps); opt.zero_grad(set_to_none=True)
    rng, step = np.random.default_rng(SEED), 0; bar = tqdm(total=steps, desc=f'{kind}: training', unit='step')
    while step < steps:
        for micro, idx in enumerate(rng.permutation(len(groups)), 1):
            g = groups[int(idx)]; x = {k:v.to(device) for k,v in inputs(tok, g, special).items()}
            y = torch.tensor(g.relevance.to_numpy(dtype=np.float32), device=device)
            loss = loss_fn(model(**x).scores.view(-1), y, kind) / ACCUM; loss.backward()
            if micro % ACCUM and micro != len(groups): continue
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step(); sch.step(); opt.zero_grad(set_to_none=True); step += 1
            bar.update(1); bar.set_postfix(loss=f'{loss.item()*ACCUM:.4f}')
            if step >= steps: break
    bar.close(); out = infer(model, tok, test, special, f'{kind}: test inference')
    del model, tok, opt, sch; gc.collect(); torch.cuda.empty_cache(); return out

def query_rank(frame, scores):
    work = frame[['query_id','id']].copy(); work['score'] = scores; out = np.empty(len(work), dtype=np.float32)
    for _, g in work.groupby('query_id', sort=False):
        idx = g.sort_values(['score','id'], ascending=[True,False], kind='mergesort').index
        out[idx] = np.arange(1, len(g)+1, dtype=np.float32) / len(g)
    return out



## 5. Загрузка данных, обучение и `submission.csv`


In [ ]:
if not torch.cuda.is_available(): raise RuntimeError('CUDA is required for this final solution.')
free, total = (x/2**30 for x in torch.cuda.mem_get_info())
print(f'GPU memory available: {free:.1f} / {total:.1f} GiB')
DATA = find_data(); train = pd.read_csv(DATA/'train.csv').reset_index(drop=True); test = pd.read_csv(DATA/'test.csv').reset_index(drop=True)
need_train, need_test = {'id','query_id','query','title','text','relevance'}, {'id','query_id','query','title','text'}
if not need_train <= set(train) or not need_test <= set(test): raise ValueError('Unexpected competition CSV schema.')
print(f'GPU: {torch.cuda.get_device_name(0)}; data={DATA}; train={train.shape}; test={test.shape}')
# Full-data equivalents of the selected 700- and 600-step, 80%-fit checkpoints.
mixed = fit_predict('mixed', 875)
lambda_ndcg = fit_predict('lambda_ndcg10', 750)
prediction = .5 * query_rank(test, mixed) + .5 * query_rank(test, lambda_ndcg)
submission = pd.DataFrame({'id':test.id, 'prediction':prediction})
if len(submission) != len(test) or submission.id.duplicated().any() or submission.prediction.isna().any(): raise RuntimeError('Invalid submission.')
output_dir = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
output_path = output_dir / 'submission.csv'
submission.to_csv(output_path, index=False)
print(f'Wrote {output_path} ({len(submission)} rows)')
display(submission.head())
